In [1]:
import os
import pickle

import numpy as np
from tqdm import tqdm
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

base_model = InceptionV3(weights='imagenet')
feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.layers[-2].output
)

def extract(directory=r"C:\Users\komal\Documents\pbl\archive\Images", output_path='features.pkl'):
    if not os.path.exists(directory):
        print(f"Directory '{directory}' not found.")
        return

    image_files = [
        f for f in os.listdir(directory)
        if os.path.splitext(f)[1].lower() in IMAGE_EXTS
    ]

    if not image_files:
        print(f"No image files found in '{directory}'.")
        return

    print(f"\nFound {len(image_files)} images. Extracting features...\n")
    features = {}

    for img_name in tqdm(image_files, desc='Extracting'):
        img_path = os.path.join(directory, img_name)
        try:
            image   = load_img(img_path, target_size=(299, 299))
            image   = img_to_array(image)
            image   = np.expand_dims(image, axis=0)
            image   = preprocess_input(image)
            feature = feature_extractor.predict(image, verbose=0)
            img_id  = os.path.splitext(img_name)[0]
            features[img_id] = feature[0]
        except Exception as e:
            print(f"\nSkipping '{img_name}': {e}")

    pickle.dump(features, open(output_path, 'wb'))
    print(f"\n {len(features)} feature vectors saved → '{output_path}'")

extract()


Found 8091 images. Extracting features...



Extracting:   0%|                                                                             | 0/8091 [00:00<?, ?it/s]

Extracting: 100%|██████████████████████████████████████████████████████████████████| 8091/8091 [45:04<00:00,  2.99it/s]



 8091 feature vectors saved → 'features.pkl'


In [3]:
import os
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

def clean_text(captions_path=r'C:\Users\komal\Documents\pbl\archive\captions.txt'):
    if not os.path.exists(captions_path):
        print(f" '{captions_path}' not found.")
        return

    mapping = {}
    with open(captions_path, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',', 1)
            if len(parts) < 2:
                continue
            img_id  = parts[0].split('.')[0].split('#')[0]
            caption = parts[1].lower().strip()
            if img_id not in mapping:
                mapping[img_id] = []
            mapping[img_id].append(f"startseq {caption} endseq")

    if not mapping:
        print(" No captions parsed. Check the format of captions.txt.")
        return

    all_captions = [cap for caps in mapping.values() for cap in caps]
    tokenizer    = Tokenizer()
    tokenizer.fit_on_texts(all_captions)
    max_length   = max(len(cap.split()) for cap in all_captions)

    pickle.dump(tokenizer, open('tokenizer.pkl', 'wb'))
    pickle.dump(mapping,   open('mapping.pkl',   'wb'))
    with open('max_length.txt', 'w') as f:
        f.write(str(max_length))

    vocab_size = len(tokenizer.word_index) + 1
    print(f" tokenizer.pkl  saved  ({vocab_size} vocab tokens)")
    print(f" mapping.pkl    saved  ({len(mapping)} images)")
    print(f" max_length.txt saved  (max length = {max_length})")

clean_text()

 tokenizer.pkl  saved  (8496 vocab tokens)
 mapping.pkl    saved  (8091 images)
 max_length.txt saved  (max length = 40)


In [4]:
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.models import Model

def define_model(vocab_size, max_length):

    inputs1 = Input(shape=(2048,),       name='image_input')
    fe1     = Dropout(0.5)(inputs1)
    fe2     = Dense(512, activation='relu', name='image_dense')(fe1)

    # Decoder: caption sequence branch
    inputs2 = Input(shape=(max_length,), name='sequence_input')
    se1     = Embedding(vocab_size, 512, mask_zero=True, name='embedding')(inputs2)
    se2     = Dropout(0.5)(se1)
    se3     = LSTM(512, name='lstm')(se2)

    # Merge
    decoder1 = add([fe2, se3],                          name='merge')
    decoder2 = Dense(512, activation='relu',            name='decoder_dense')(decoder1)
    outputs  = Dense(vocab_size, activation='softmax',  name='output')(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs, name='VisionVoice')
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

test_model = define_model(vocab_size=1000, max_length=36)
test_model.summary()

Model: "VisionVoice"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)   │ (None, 36)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ image_input (InputLayer)      │ (None, 2048)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 36, 512)           │         512,000 │ sequence_input[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 2048)              │               0 │ image_input[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 36, 512)           │               0 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 36)                │               0 │ sequence_input[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ image_dense (Dense)           │ (None, 512)               │       1,049,088 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 512)               │       2,099,200 │ dropout_1[0][0],           │
│                               │                           │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ merge (Add)                   │ (None, 512)               │               0 │ image_dense[0][0],         │
│                               │                           │                 │ lstm[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_dense (Dense)         │ (None, 512)               │         262,656 │ merge[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output (Dense)                │ (None, 1000)              │         513,000 │ decoder_dense[0][0]        │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,435,944 (16.92 MB)

 Trainable params: 4,435,944 (16.92 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:

import os
import pickle
import random

import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Embedding, Input, LSTM, add
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print("=" * 50)
print("       VisionVoice — Training")
print("=" * 50)

for fname in ['features.pkl', 'tokenizer.pkl', 'mapping.pkl', 'max_length.txt']:
    if not os.path.exists(fname):
        print(f" '{fname}' not found. Run Steps 1 & 2 first.")
        raise FileNotFoundError(fname)

features   = pickle.load(open('features.pkl',  'rb'))
tokenizer  = pickle.load(open('tokenizer.pkl', 'rb'))
mapping    = pickle.load(open('mapping.pkl',   'rb'))
vocab_size = len(tokenizer.word_index) + 1

with open('max_length.txt', 'r') as f:
    max_length = int(f.read().strip())

print(f"\n   Vocab size  : {vocab_size}")
print(f"   Max length  : {max_length}")
print(f"   Images      : {len(mapping)}")
print(f"   Features    : {len(features)}\n")

def data_generator(mapping, features, tokenizer, max_length, vocab_size, batch_size):
    keys = list(mapping.keys())
    while True:
        random.shuffle(keys)
        for i in range(0, len(keys), batch_size):
            batch_keys = keys[i:i + batch_size]
            X1, X2, y = [], [], []
            for key in batch_keys:
                if key not in features:
                    continue
                feature  = features[key]
                captions = mapping[key]
                for caption in captions:
                    seq = tokenizer.texts_to_sequences([caption])[0]
                    for j in range(1, len(seq)):
                        in_seq  = pad_sequences([seq[:j]], maxlen=max_length)[0]
                        out_seq = to_categorical([seq[j]], num_classes=vocab_size)[0]
                        X1.append(feature)
                        X2.append(in_seq)
                        y.append(out_seq)
            if not X1:
                continue
            yield (np.array(X1), np.array(X2)), np.array(y)

CHECKPOINT_PATH = 'vision_voice_model.keras'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Resuming from '{CHECKPOINT_PATH}'...")
    model = load_model(CHECKPOINT_PATH)
else:
    print("Building model from scratch...")
    inputs1  = Input(shape=(2048,),       name='image_input')
    fe1      = Dropout(0.5)(inputs1)
    fe2      = Dense(512, activation='relu', name='image_dense')(fe1)
    inputs2  = Input(shape=(max_length,), name='sequence_input')
    se1      = Embedding(vocab_size, 512, mask_zero=True, name='embedding')(inputs2)
    se2      = Dropout(0.5)(se1)
    se3      = LSTM(512, name='lstm')(se2)
    decoder1 = add([fe2, se3])
    decoder2 = Dense(512, activation='relu')(decoder1)
    outputs  = Dense(vocab_size, activation='softmax')(decoder2)
    model    = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

callbacks = [
    ModelCheckpoint(CHECKPOINT_PATH, monitor='loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='loss', patience=5, restore_best_weights=True, verbose=1),
]

BATCH_SIZE = 32
EPOCHS     = 30
steps      = max(1, len(mapping) // BATCH_SIZE)

print(f"\nStarting training  |  Target loss < 3.0")
print(f"Epochs: {EPOCHS}  |  Steps/epoch: {steps}  |  Batch: {BATCH_SIZE}\n")

model.fit(
    data_generator(mapping, features, tokenizer, max_length, vocab_size, BATCH_SIZE),
    steps_per_epoch=steps,
    epochs=EPOCHS,
    verbose=1,
    callbacks=callbacks,
)

print(f"\n Training complete. Best model saved → '{CHECKPOINT_PATH}'")

       VisionVoice — Training

   Vocab size  : 8496
   Max length  : 40
   Images      : 8091
   Features    : 8091

Building model from scratch...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)   │ (None, 40)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ image_input (InputLayer)      │ (None, 2048)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 40, 512)           │       4,349,952 │ sequence_input[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 2048)              │               0 │ image_input[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_3 (Dropout)           │ (None, 40, 512)           │               0 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal_1 (NotEqual)        │ (None, 40)                │               0 │ sequence_input[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ image_dense (Dense)           │ (None, 512)               │       1,049,088 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 512)               │       2,099,200 │ dropout_3[0][0],           │
│                               │                           │                 │ not_equal_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 512)               │               0 │ image_dense[0][0],         │
│                               │                           │                 │ lstm[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 512)               │         262,656 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 8496)              │       4,358,448 │ dense[0][0]                │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 12,119,344 (46.23 MB)

 Trainable params: 12,119,344 (46.23 MB)

 Non-trainable params: 0 (0.00 B)


Starting training  |  Target loss < 3.0
Epochs: 30  |  Steps/epoch: 252  |  Batch: 32

Epoch 1/30
252/252 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - loss: 5.1959 
Epoch 1: loss improved from None to 4.37064, saving model to vision_voice_model.keras

Epoch 1: finished saving model to vision_voice_model.keras
252/252 ━━━━━━━━━━━━━━━━━━━━ 3461s 14s/step - loss: 4.3706 - learning_rate: 0.0010
Epoch 2/30
252/252 ━━━━━━━━━━━━━━━━━━━━ 0s 15s/step - loss: 3.3300 
Epoch 2: loss improved from 4.37064 to 3.26547, saving model to vision_voice_model.keras

Epoch 2: finished saving model to vision_voice_model.keras
252/252 ━━━━━━━━━━━━━━━━━━━━ 3707s 15s/step - loss: 3.2655 - learning_rate: 0.0010
Epoch 3/30
252/252 ━━━━━━━━━━━━━━━━━━━━ 0s 16s/step - loss: 2.9552 
Epoch 3: loss improved from 3.26547 to 2.94063, saving model to vision_voice_model.keras

Epoch 3: finished saving model to vision_voice_model.keras
252/252 ━━━━━━━━━━━━━━━━━━━━ 4003s 16s/step - loss: 2.9406 - learning_rate: 0.0010
Epoch 4/30
 92/

ResourceExhaustedError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
MemoryError: Unable to allocate 121. MiB for an array with shape (1871, 8496) and data type float64
Traceback (most recent call last):

  File "C:\Users\komal\anaconda3\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "C:\Users\komal\anaconda3\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "C:\Users\komal\anaconda3\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "C:\Users\komal\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\generator_data_adapter.py", line 54, in get_tf_iterator
    for batch in self.generator():
                 ~~~~~~~~~~~~~~^^

  File "C:\Users\komal\AppData\Local\Temp\i

In [ ]:
import pyttsx3

def speak_caption(caption, rate=150):
    clean = caption.replace('startseq', '').replace('endseq', '').strip()
    if not clean:
        print("  Nothing to speak — caption is empty.")
        return
    print(f" VisionVoice is saying: {clean}")
    engine = pyttsx3.init()
    engine.setProperty('rate', rate)
    engine.say(clean)
    engine.runAndWait()


In [ ]:
import cv2

def capture_image(save_path='data/Images/captured.jpg'):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print(" Could not open webcam. Try VideoCapture(1) instead.")
        return None

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print("\n Camera is ON!")
    print("    Press SPACEBAR to start the 3-second countdown and capture")
    print("    Press  Q  to quit without capturing\n")

    captured_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            print(" Failed to read from camera.")
            break

        display = frame.copy()
        cv2.putText(display, "VisionVoice  |  Live Camera",
                    (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(display, "SPACE = Capture   Q = Quit",
                    (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.imshow("VisionVoice Camera", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord(' '):
            for count in range(3, 0, -1):
                for _ in range(20):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    countdown = frame.copy()
                    cx = countdown.shape[1] // 2 - 40
                    cy = countdown.shape[0] // 2 + 40
                    cv2.putText(countdown, str(count), (cx, cy),
                                cv2.FONT_HERSHEY_SIMPLEX, 4, (0, 255, 0), 6, cv2.LINE_AA)
                    cv2.imshow("VisionVoice Camera", countdown)
                    cv2.waitKey(50)
            ret, frame = cap.read()
            if ret:
                captured_frame = frame.copy()
                white = frame.copy()
                white[:] = (255, 255, 255)
                cv2.imshow("VisionVoice Camera", white)
                cv2.waitKey(200)
                print(" Image captured!")
            break

        elif key in (ord('q'), ord('Q')):
            print(" Camera closed without capturing.")
            break

    cap.release()
    cv2.destroyAllWindows()

    if captured_frame is not None:
        cv2.imwrite(save_path, captured_frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        print(f" Image saved → {save_path}")
        return save_path
    return None

print(" capture_image() ready")

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

print("=" * 50)
print("      VisionVoice — Live Demo")
print("=" * 50)

print("\nLoading model...")
try:
    demo_model = load_model('vision_voice_model.keras')
except Exception:
    demo_model = load_model('vision_voice_model.keras',
                            custom_objects={"NotEqual": tf.math.not_equal},
                            compile=False)
print(" Model loaded!")

# Load tokenizer + max_length
demo_tokenizer = pickle.load(open('tokenizer.pkl', 'rb'))
with open('max_length.txt', 'r') as f:
    demo_max_length = int(f.read().strip())
print(f"   Vocab size : {len(demo_tokenizer.word_index) + 1}")
print(f"   Max length : {demo_max_length}")

# Load InceptionV3 feature extractor
print("\nLoading InceptionV3 feature extractor...")
iv3_base      = InceptionV3(weights='imagenet')
demo_feat_mdl = Model(inputs=iv3_base.input, outputs=iv3_base.layers[-2].output)
print(" Feature extractor ready!\n")

def predict_caption(image_path):
    img = load_img(image_path, target_size=(299, 299))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    feature  = demo_feat_mdl.predict(img, verbose=0)
    in_text  = 'startseq'
    for _ in range(demo_max_length):
        sequence = demo_tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=demo_max_length)
        yhat     = demo_model.predict([feature, sequence], verbose=0)
        yhat     = np.argmax(yhat)
        word     = next(
            (w for w, idx in demo_tokenizer.word_index.items() if idx == yhat),
            None
        )
        if word is None or word == 'endseq':
            break
        in_text += ' ' + word
    return in_text

def run_demo():
    print("\n" + "─" * 45)
    img_path = capture_image(save_path='data/Images/captured.jpg')
    if img_path is None:
        print("No image captured.")
        return

    print("\n AI is analysing the scene...")
    raw_caption   = predict_caption(img_path)
    clean_caption = raw_caption.replace('startseq', '').strip()
    print(f"\n Generated Caption: {clean_caption}\n")

    plt.figure(figsize=(7, 5))
    plt.imshow(load_img(img_path))
    plt.title(f"VisionVoice Prediction:\n{clean_caption}", fontsize=13, pad=12)
    plt.axis('off')
    plt.tight_layout()
    print("  Close the image window to hear the audio description.")
    plt.show()

    speak_caption(clean_caption)

run_demo()

In [ ]:
run_demo()